<a href="https://colab.research.google.com/github/codebysumit/cryptography-algorithms/blob/master/notebooks/columnar_transposition_cipher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Columnar Transposition Cipher

## History
The Columnar Transposition Cipher has been used since the 19th century, and it remained popular well into the 20th century because it is easy to do by hand with just pen, paper, and a grid. It was used by both sides in World War 1, and a more advanced double version of it (encrypting the output a second time with a second keyword) was used by German forces in World War 2.

## What is Columnar Transposition Cipher?
Just like the Rail Fence Cipher, this is a **transposition cipher**. It never changes what a character is, it only changes the order the characters appear in. But instead of a zigzag pattern, Columnar Transposition writes the plaintext into a **grid**, row by row, and then reads the grid back out **column by column**, in an order controlled by a keyword.

The number of columns in the grid is the same as the length of the keyword. Since the keyword can be any length, the **track size (number of columns) is variable**, exactly like the rail count was variable in Rail Fence Cipher.

In this implementation, we support the full **printable ASCII range**, from space (` `) to tilde (`~`), which is ASCII value 32 to 126. Since this cipher only rearranges characters instead of transforming them, every character in that range, letters, digits, spaces, and punctuation, is treated exactly the same way.

## Cryptography Algorithm

### Constants
*   $S = 32$ (Start of printable ASCII range)
*   $E = 126$ (End of printable ASCII range)
*   $K$ = the keyword, its length decides the **number of columns** ($n$)
*   $L$ = length of the plaintext

### 1. Deciding the Column Order
Every character of the keyword gets a **rank**, based on alphabetical order. If two characters in the keyword are the same, the one that appears first (left to right) gets the smaller rank. This rank tells us in what order the columns will be read out.

For example, with the keyword **HACK**:

```
Keyword:  H  A  C  K
Rank:     2  0  1  3
```

A is the earliest letter alphabetically, so it gets rank 0. C comes next, rank 1. H comes after that, rank 2. K is last, rank 3. This means the columns will be read in this order: the column under **A**, then the column under **C**, then the column under **H**, then the column under **K**.

### 2. Filling the Grid
The plaintext is written into the grid **row by row**, left to right, top to bottom, using $n$ columns (the length of the keyword). If the plaintext length is not an exact multiple of $n$, the very last row will simply be shorter, since this implementation does not use any padding characters. This keeps every single character of the printable ASCII range usable in the message, without needing a special filler character that might clash with real message content.

### 3. Encryption
Once the grid is filled, the ciphertext is built by reading the grid **column by column**, following the rank order worked out in Step 1, not the original left to right column order.

### 4. Decryption
Decryption rebuilds the grid in reverse.

1.  Work out the same column ranks from the keyword.
2.  Work out how many rows the grid had, and exactly how many characters belong to each column (some columns may have one extra character if the last row was not full).
3.  Slice the ciphertext into chunks matching each column's length, placing each chunk into its correct column, following the rank order.
4.  Read the grid back out **row by row** to recover the original plaintext.

### 5. Fully Worked Example (By Hand)
Let's encrypt **ATTACKATDAWN** using the keyword **HACK**.

**Step 1: Work out the column ranks**, as shown above: H=2, A=0, C=1, K=3.

**Step 2: Fill a 4 column grid, row by row.** The message is exactly 12 characters, and the keyword has 4 letters, so this fits perfectly into 3 full rows with no leftover characters.

```
Column:   H  A  C  K
Rank:     2  0  1  3

Row 1:    A  T  T  A
Row 2:    C  K  A  T
Row 3:    D  A  W  N
```

**Step 3: Read the columns out in rank order (0, 1, 2, 3), not left to right.**

*   Rank 0 is column A: reading down gives **T K A**
*   Rank 1 is column C: reading down gives **T A W**
*   Rank 2 is column H: reading down gives **A C D**
*   Rank 3 is column K: reading down gives **A T N**

**Step 4: Concatenate in that order.**

**Ciphertext = TKATAWACDATN**

To decrypt, we would work out the same ranks, realise this message splits evenly into 4 columns of 3 characters each, slice the ciphertext back into those 4 chunks of 3, place each chunk under its correct keyword letter, and read the grid back out row by row to get **ATTACKATDAWN**.

### Key Requirements
*   The key is a **keyword**, and its length sets the **number of columns**.
*   The keyword should be **at least 2 characters** long. A 1 column grid does not scramble anything.
*   The keyword can repeat letters, ties are broken by the letter's original left to right position.
*   A longer keyword means more columns, which usually spreads the plaintext out more and scrambles it harder.

### 1. Import Dependencies

In [ ]:
import random
import string
import math

### 2. Helper Utilities

In [ ]:
START_ASCII = 32
END_ASCII = 126

def validate_printable_text(text: str) -> None:
    # Columnar Transposition only rearranges characters, but every character must
    # still fall inside our supported printable ASCII range
    for ch in text:
        code = ord(ch)
        if not (START_ASCII <= code <= END_ASCII):
            raise ValueError(
                f"Character {ch!r} (ASCII {code}) is outside the supported range "
                f"{START_ASCII}-{END_ASCII}."
            )

def build_column_order(keyword: str) -> list:
    # rank[i] = the reading order rank of column i, based on alphabetical order of the keyword
    # ties (repeated letters) are broken by original left-to-right position
    indexed = list(enumerate(keyword))
    sorted_indexed = sorted(indexed, key=lambda pair: (pair[1], pair[0]))

    order = [0] * len(keyword)
    for rank, (original_index, ch) in enumerate(sorted_indexed):
        order[original_index] = rank

    return order

### 3. Generate a Random Key (Keyword)

In [ ]:
def generate_random_key(min_length: int = 4, max_length: int = 10) -> str:
    # random keyword length gives a variable track size (variable number of columns)
    length = random.randint(min_length, max_length)
    return "".join(random.choice(string.ascii_uppercase) for _ in range(length))

### 4. Encryption

In [ ]:
def encrypt(text: str, keyword: str) -> str:
    validate_printable_text(text)

    num_cols = len(keyword)
    if num_cols < 2:
        raise ValueError("'keyword' must be at least 2 characters long.")

    order = build_column_order(keyword)
    length = len(text)

    num_rows = math.ceil(length / num_cols)
    remainder = length % num_cols
    if remainder == 0:
        remainder = num_cols  # every column is completely full, none are short

    # fill the grid row by row, storing each column's characters separately
    grid_columns = [[] for _ in range(num_cols)]
    index = 0
    for row in range(num_rows):
        cols_in_this_row = num_cols if row < num_rows - 1 or remainder == num_cols else remainder
        for col in range(cols_in_this_row):
            grid_columns[col].append(text[index])
            index += 1

    # figure out which column to read first, second, third... based on rank
    column_by_rank = [None] * num_cols
    for col_index, rank in enumerate(order):
        column_by_rank[rank] = col_index

    cipher_text = ""
    for col_index in column_by_rank:
        cipher_text += "".join(grid_columns[col_index])

    return cipher_text

### 5. Decryption

In [ ]:
def decrypt(cipher_text: str, keyword: str) -> str:
    validate_printable_text(cipher_text)

    num_cols = len(keyword)
    order = build_column_order(keyword)
    length = len(cipher_text)

    num_rows = math.ceil(length / num_cols)
    remainder = length % num_cols
    if remainder == 0:
        remainder = num_cols

    # columns 0..remainder-1 (by original position) got the extra character in the last row
    column_lengths = [num_rows if col < remainder else num_rows - 1 for col in range(num_cols)]

    column_by_rank = [None] * num_cols
    for col_index, rank in enumerate(order):
        column_by_rank[rank] = col_index

    # slice the ciphertext into columns, in the rank order they were originally read out
    grid_columns = [None] * num_cols
    position = 0
    for col_index in column_by_rank:
        col_len = column_lengths[col_index]
        grid_columns[col_index] = list(cipher_text[position:position + col_len])
        position += col_len

    # read the grid back out row by row
    plain_text = []
    column_pointers = [0] * num_cols
    for row in range(num_rows):
        cols_in_this_row = num_cols if row < num_rows - 1 or remainder == num_cols else remainder
        for col in range(cols_in_this_row):
            plain_text.append(grid_columns[col][column_pointers[col]])
            column_pointers[col] += 1

    return "".join(plain_text)

### 6. Verify the Hand Worked Example in Code

In [ ]:
hand_keyword = "HACK"
hand_plaintext = "ATTACKATDAWN"

print("Column order (rank per column):", build_column_order(hand_keyword))

hand_cipher = encrypt(hand_plaintext, hand_keyword)
hand_decrypted = decrypt(hand_cipher, hand_keyword)

print(f"Plaintext: {hand_plaintext}")
print(f"Encrypted: {hand_cipher}  (should match TKATAWACDATN from the hand example)")
print(f"Decrypted: {hand_decrypted}")

Column order (rank per column): [2, 0, 1, 3]
Plaintext: ATTACKATDAWN
Encrypted: TKATAWACDATN  (should match TKATAWACDATN from the hand example)
Decrypted: ATTACKATDAWN


### 7. Example usage

In [ ]:
key = generate_random_key()
print(f"Generated Random Key (keyword): {key}")

Generated Random Key (keyword): GOXHCYAIV


In [ ]:
plaintext = """TOP secret Massage! Agent 101, visit Area 51 (37d14'0\"N 115d48'30\"W)."""
print(f"Original Plain Text: {plaintext}")

cipher_text = encrypt(plaintext, key)
print("Encrypted:", cipher_text)

decrypted_text = decrypt(cipher_text, key)
print("Decrypted:", decrypted_text)

match = plaintext == decrypted_text
print(f"Verification Match:{match}")

Original Plain Text: TOP secret Massage! Agent 101, visit Area 51 (37d14'0"N 115d48'30"W).
Encrypted: cats5'4sseva15)Tt!0 (N3 ag ed1Wrg i108O  1A3 0ee1t "'PMA,r71"esni 4d.
Decrypted: TOP secret Massage! Agent 101, visit Area 51 (37d14'0"N 115d48'30"W).
Verification Match:True


### 8. Trying Different Track Sizes on the Same Message

In [ ]:
for keyword_length in [2, 3, 4, 5, 8]:
    test_key = generate_random_key(keyword_length, keyword_length)
    cipher_text = encrypt(plaintext, test_key)
    decrypted_text = decrypt(cipher_text, test_key)
    print(f"Keyword={test_key:<8} Cipher: {cipher_text}")
    print(f"           Match: {decrypted_text == plaintext}\n")

Keyword=JX       Cipher: TPsce asg!Aet11 ii ra5 3d40N154'0W.O ertMsae gn 0,vstAe 1(71'" 1d83")
           Match: True

Keyword=ELB      Cipher: PeeMseAn1,itr  74"1d'".T ctaa!gt0 s e5(d'N143WOsr sg e 1viAa1310 580)
           Match: True

Keyword=ZFGC     Cipher:  rMa n0vte17' d3)Oetseg ,sA (1"18"Pc s!e1 ir534N5'WTseagAt1i a d0140.
           Match: True

Keyword=MAFGD    Cipher: OcMgg1v  3'18Wsts t,ie 1Nd0Praee0iA5701') es!n1sr1d"53.Te aA  ta(4 4"
           Match: True

Keyword=TPWFCZGI Cipher: saA1  04. M 0t1'd)cse r3N'ranve7 3Ote s 11"Tegtiad10P !1i545Wesg,A("8
           Match: True

